<a href="https://colab.research.google.com/github/stefkong1982/netology.ru/blob/Master/%D0%92%D0%B7%D0%B0%D0%B8%D0%BC_%D0%9E%D0%B3%D1%80_100_ml_ozon_recsys_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# OZON RecSys Baseline - Рекомендательная система для категории Apparel
Этот ноутбук содержит базовое решение для задачи предсказания следующей покупки пользователя в категории одежды, обуви и аксессуаров.

## Задача
- Предсказать топ-100 товаров для каждого пользователя из тестовой выборки
- Метрика оценки: NDCG@100
- Данные: ~38GB в формате parquet, 1.6B взаимодействий, 19M заказов


In [1]:
# === 1. Монтируем Google Drive, задаём пути к данным (структура Colab/Яндекс) ===
from google.colab import drive
drive.mount('/content/drive')

# Шаг 1. Импорты и пути (Colab/локально, без лишних библиотек)
import pandas as pd
import numpy as np
from tqdm import tqdm
from pathlib import Path
from collections import defaultdict, Counter # Добавлены для подсчета популярности
import glob

# Пути к данным (замените на свои)
ORDERS_PATH = '/content/drive/MyDrive/Colab Notebooks/e_cup_2025_project/data/2_raw/extra_ml_ozon_recsys_train/archive_extra_orders_data/final_apparel_orders_data_07'
ORDERS_PATH2 = '/content/drive/MyDrive/Colab Notebooks/e_cup_2025_project/data/raw/ml_ozon_recsys_train_final_apparel_orders_data'
# НОВЫЙ ПУТЬ: Добавляем путь к данным взаимодействий за нужный период
TRACKER_PATH = '/content/drive/MyDrive/Colab Notebooks/e_cup_2025_project/data/2_raw/extra_ml_ozon_recsys_train/archive_extra_tracker_data/final_apparel_tracker_data_08'
TEST_PATH = '/content/drive/MyDrive/Colab Notebooks/e_cup_2025_project/data/2_raw/ml_ozon_recsys_test'

# Шаг 2. Загрузка всех заказов (train)
def load_orders():
    print("Загружаем тренировочные данные заказов...")
    orders = []
    for path in [ORDERS_PATH, ORDERS_PATH2]:
        for f in Path(path).rglob('*.parquet'):
            orders.append(pd.read_parquet(f))
    df = pd.concat(orders, ignore_index=True)
    # Обогащаем created_date, если нужно
    if 'created_date' in df.columns and df['created_date'].isna().sum():
        df['created_timestamp'] = pd.to_datetime(df['created_timestamp'])
        df['created_date'] = df['created_date'].fillna(df['created_timestamp'].dt.date)
        df['created_date'] = pd.to_datetime(df['created_date'])
    elif 'created_date' not in df.columns and 'created_timestamp' in df.columns:
        # Если created_date вообще отсутствует, создаем её
        df['created_timestamp'] = pd.to_datetime(df['created_timestamp'])
        df['created_date'] = df['created_timestamp'].dt.date
        df['created_date'] = pd.to_datetime(df['created_date'])
    print(f"Загружено заказов: {len(df):,}")
    return df

orders_df = load_orders()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Загружаем тренировочные данные заказов...
Загружено заказов: 20,362,338


In [2]:
print("=== 📘 ТРЕНИРОВОЧНЫЕ ДАННЫЕ ЗАКАЗОВ (orders_df) ===")
print("\n👉 Первые 2 строки с типами данных:")
print(orders_df.head(2).to_markdown(tablefmt="grid"))

print("\n📊 Схема данных:")
for col in orders_df.columns:
    print(f"  • {col}: {orders_df[col].dtype}")

=== 📘 ТРЕНИРОВОЧНЫЕ ДАННЫЕ ЗАКАЗОВ (orders_df) ===

👉 Первые 2 строки с типами данных:
+----+-----------+-----------+----------------------------+------------------+-------------------------+---------------------+
|    |   item_id |   user_id | created_timestamp          | last_status      | last_status_timestamp   | created_date        |
+====+===========+===========+============================+==================+=========================+=====================+
|  0 | 332361399 |      2841 | 2025-07-14 08:38:22.250000 | proccesed_orders | 2025-07-14 11:35:22     | 2025-07-14 00:00:00 |
+----+-----------+-----------+----------------------------+------------------+-------------------------+---------------------+
|  1 | 153141296 |      3681 | 2025-07-14 09:13:56.410000 | proccesed_orders | 2025-07-14 10:47:53     | 2025-07-14 00:00:00 |
+----+-----------+-----------+----------------------------+------------------+-------------------------+---------------------+

📊 Схема данных:
  • ite

In [3]:
# Шаг 2.5. Загрузка взаимодействий (tracker)
def load_tracker():
    print("Загружаем тренировочные данные взаимодействий...")
    tracker_data = []
    # Используем rglob для рекурсивного поиска всех .parquet файлов
    # в директории TRACKER_PATH и её поддиректориях
    for f in Path(TRACKER_PATH).rglob('*.parquet'):
        tracker_data.append(pd.read_parquet(f))

    if tracker_data:
        df = pd.concat(tracker_data, ignore_index=True)
        print(f"Загружено взаимодействий: {len(df):,}")
        return df
    else:
        print("Файлы взаимодействий не найдены.")
        return pd.DataFrame() # Возвращаем пустой DataFrame

tracker_df = load_tracker()

Загружаем тренировочные данные взаимодействий...
Загружено взаимодействий: 116,552,409


In [4]:
print("\n\n=== 📙 ДАННЫЕ ВЗАИМОДЕЙСТВИЙ (tracker_df) ===")
print("\n👉 Первые 2 строки с типами данных:")
print(tracker_df.head(2).to_markdown(tablefmt="grid"))

print("\n📊 Схема данных:")
for col in tracker_df.columns:
    print(f"  • {col}: {tracker_df[col].dtype}")



=== 📙 ДАННЫЕ ВЗАИМОДЕЙСТВИЙ (tracker_df) ===

👉 Первые 2 строки с типами данных:
+----+-----------------+-----------+-----------+---------------------+---------------+
|    | action_widget   |   item_id |   user_id | timestamp           | action_type   |
+====+=================+===========+===========+=====================+===============+
|  0 | pdp             |    996252 |   3542470 | 2025-07-08 23:29:23 | to_cart       |
+----+-----------------+-----------+-----------+---------------------+---------------+
|  1 | pdp             |   4127285 |   3267450 | 2025-07-09 14:51:02 | to_cart       |
+----+-----------------+-----------+-----------+---------------------+---------------+

📊 Схема данных:
  • action_widget: object
  • item_id: int32
  • user_id: int32
  • timestamp: datetime64[ns]
  • action_type: object


In [5]:
# Шаг 3. Загрузка тестовых пользователей
def load_test_users():
    print("Загружаем тестовых пользователей...")
    test_files = glob.glob(f'{TEST_PATH}/*.parquet')
    users = set()
    for f in tqdm(test_files, desc="Обработка тестовых файлов"):
        df_part = pd.read_parquet(f)
        if 'user_id' in df_part.columns:
            users.update(df_part['user_id'].unique())
    print(f"Найдено уникальных тестовых пользователей: {len(users):,}")
    return list(users)

test_users = load_test_users()

Загружаем тестовых пользователей...


Обработка тестовых файлов: 100%|██████████| 1/1 [00:00<00:00,  7.71it/s]

Найдено уникальных тестовых пользователей: 470,347


In [6]:
print("=== 📗 ТЕСТОВЫЕ ПОЛЬЗОВАТЕЛИ ===")
test_users_df = pd.DataFrame({'user_id': test_users})
print(f"📊 Размер: {len(test_users_df):,} строк")
print("\n👉 Первые 2 строки:")
print(test_users_df.head(2).to_string())
print("\n📋 Колонки и типы данных:")
for col in test_users_df.columns:
    print(f"  • {col:20} {test_users_df[col].dtype}")

=== 📗 ТЕСТОВЫЕ ПОЛЬЗОВАТЕЛИ ===
📊 Размер: 470,347 строк

👉 Первые 2 строки:
   user_id
0        1
1  3145730

📋 Колонки и типы данных:
  • user_id              int32


In [7]:
print("\n\nАНАЛИЗ ЗАКАЗОВ")
print("=" * 50)
print(f"Общее количество заказов: {len(orders_df):,}")
print(f"Уникальных пользователей: {orders_df['user_id'].nunique():,}")
print(f"Уникальных товаров: {orders_df['item_id'].nunique():,}")

if 'created_date' in orders_df.columns:
    min_date = orders_df['created_date'].min().date()
    max_date = orders_df['created_date'].max().date()
    print(f"Период данных: {min_date} - {max_date}")

if 'last_status' in orders_df.columns:
    print("\nРаспределение статусов заказов:")
    status_counts = orders_df['last_status'].value_counts()
    status_counts_pct = orders_df['last_status'].value_counts(normalize=True) * 100
    for status, count in status_counts.items():
        pct = status_counts_pct[status]
        print(f"  {status}: {count:,} ({pct:.1f}%)")




АНАЛИЗ ЗАКАЗОВ
Общее количество заказов: 20,362,338
Уникальных пользователей: 842,254
Уникальных товаров: 4,679,218
Период данных: 2025-01-01 - 2025-07-15

Распределение статусов заказов:
  delivered_orders: 10,420,894 (51.2%)
  canceled_orders: 8,420,631 (41.4%)
  proccesed_orders: 1,520,813 (7.5%)


In [8]:
print("\n\nАНАЛИЗ ВЗАИМОДЕЙСТВИЙ")
print("=" * 50)
print(f"Общее количество взаимодействий: {len(tracker_df):,}")
print(f"Уникальных пользователей: {tracker_df['user_id'].nunique():,}")
print(f"Уникальных товаров: {tracker_df['item_id'].nunique():,}")

if 'timestamp' in tracker_df.columns:
    min_date = tracker_df['timestamp'].min().date()
    max_date = tracker_df['timestamp'].max().date()
    print(f"Период данных: {min_date} - {max_date}")

if 'action_type' in tracker_df.columns:
    print("\nРаспределение типов действий:")
    action_counts = tracker_df['action_type'].value_counts()
    action_counts_pct = tracker_df['action_type'].value_counts(normalize=True) * 100
    for action, count in action_counts.items():
        pct = action_counts_pct[action]
        print(f"  {action}: {count:,} ({pct:.1f}%)")



АНАЛИЗ ВЗАИМОДЕЙСТВИЙ
Общее количество взаимодействий: 116,552,409
Уникальных пользователей: 807,670
Уникальных товаров: 3,197,923
Период данных: 2010-01-30 - 2025-07-16

Распределение типов действий:
  page_view: 80,605,340 (69.2%)
  view_description: 17,253,961 (14.8%)
  review_view: 5,729,054 (4.9%)
  to_cart: 4,618,070 (4.0%)
  favorite: 3,455,017 (3.0%)
  remove: 3,007,598 (2.6%)
  unfavorite: 1,883,369 (1.6%)


In [9]:
# Для tracker_df по timestamp
if 'timestamp' in tracker_df.columns:
    print("\nКоличество взаимодействий по годам (на основе timestamp):")
    tracker_by_year = tracker_df['timestamp'].dt.year.value_counts().sort_index()
    for year, count in tracker_by_year.items():
        print(f"  {year}: {count:,}")


Количество взаимодействий по годам (на основе timestamp):
  2010: 1
  2024: 12
  2025: 116,552,396


In [10]:
print("\n\nАНАЛИЗ ТОВАРОВ (на основе заказов и взаимодействий)")
print("=" * 50)
# Объединим item_id из orders и tracker для более полной картины
all_item_ids = set(orders_df['item_id'].unique()).union(set(tracker_df['item_id'].unique()))
print(f"Общее количество уникальных товаров в заказах и взаимодействиях: {len(all_item_ids):,}")



АНАЛИЗ ТОВАРОВ (на основе заказов и взаимодействий)
Общее количество уникальных товаров в заказах и взаимодействиях: 4,797,217


In [11]:
print("\nТоп-10 самых популярных товаров (по количеству заказов):")
top_items_orders = orders_df['item_id'].value_counts().head(10)
for item_id, count in top_items_orders.items():
    print(f"  Товар {item_id}: {count:,} заказов")



Топ-10 самых популярных товаров (по количеству заказов):
  Товар 51974017: 13,361 заказов
  Товар 187052809: 12,384 заказов
  Товар 207631139: 8,877 заказов
  Товар 143497612: 4,096 заказов
  Товар 119105606: 3,497 заказов
  Товар 247423473: 3,188 заказов
  Товар 77696741: 2,735 заказов
  Товар 175287070: 2,725 заказов
  Товар 285009143: 2,634 заказов
  Товар 201930716: 2,624 заказов


In [12]:
# === Адаптированная функция 1: build_popularity_model (с tracker_df, обработка батчами внутри памяти) ===
def build_popularity_model(orders_df, tracker_df, period_start='2025-07-02', period_end='2025-07-15', top_k=100, chunk_size=5_000_000):
    """
    Строит список популярных товаров на основе комбинированного рейтинга
    (покупки + просмотры) в указанный период.
    Обрабатывает tracker_df по частям для экономии памяти.
    """
    import pandas as pd
    from collections import defaultdict
    import gc # Для ручной очистки памяти

    print("Строим модель популярности (с учетом просмотров, батчами)...")

    # --- 1. Обработка заказов (orders_df) ---
    popular_orders = orders_df[
        (orders_df['last_status'] == 'delivered_orders') &
        (orders_df['created_date'] >= pd.to_datetime(period_start)) &
        (orders_df['created_date'] <= pd.to_datetime(period_end))
    ].copy() # copy() для избежания SettingWithCopyWarning, если нужно
    print(f"  Отфильтровано заказов за период {period_start} - {period_end}: {len(popular_orders):,}")
    item_purchases = popular_orders['item_id'].value_counts().to_dict()
    print(f"  Уникальных купленных товаров в периоде: {len(item_purchases):,}")
    # Освобождаем память от временного фрейма
    del popular_orders
    gc.collect()

    # --- 2. Обработка просмотров (tracker_df) по частям ---
    print(f"  Обрабатываем tracker_df по частям размером ~{chunk_size:,} записей...")
    item_views = defaultdict(int) # Используем defaultdict для накопления счетчиков

    # Создаем итератор по частям tracker_df
    # np.array_split не подходит, так как tracker_df уже в памяти и большой
    # Лучше использовать iloc для нарезки
    total_rows = len(tracker_df)
    num_chunks = (total_rows // chunk_size) + 1

    for i in range(num_chunks):
        start_row = i * chunk_size
        end_row = min((i + 1) * chunk_size, total_rows)
        chunk = tracker_df.iloc[start_row:end_row]

        # Фильтрация чанка
        filtered_chunk = chunk[
            (chunk['action_type'] == 'page_view') &
            (chunk['timestamp'] >= pd.to_datetime(period_start)) &
            (chunk['timestamp'] <= pd.to_datetime(period_end))
        ]

        # Агрегация
        chunk_counts = filtered_chunk['item_id'].value_counts()
        for item_id, count in chunk_counts.items():
            item_views[item_id] += count

        # Очистка ссылок на чанк
        del chunk, filtered_chunk, chunk_counts
        if i % 10 == 0 or i == num_chunks - 1: # Промежуточный лог
             print(f"    Обработано {i+1}/{num_chunks} частей tracker_df")
        gc.collect() # Принудительная сборка мусора

    print(f"  Всего уникальных просмотренных товаров в периоде: {len(item_views):,}")

    # --- 3. Комбинирование рейтингов ---
    combined_scores = defaultdict(float)
    purchase_weight = 3.0
    view_weight = 1.0

    print("  Рассчитываем комбинированный рейтинг...")
    for item_id, count in item_purchases.items():
        combined_scores[item_id] += count * purchase_weight

    for item_id, count in item_views.items():
        combined_scores[item_id] += count * view_weight

    print(f"  Уникальных товаров с комбинированным рейтингом: {len(combined_scores):,}")

    # --- 4. Сортировка и выбор топ-K ---
    sorted_items = sorted(combined_scores.items(), key=lambda x: x[1], reverse=True)
    top_items = [item_id for item_id, score in sorted_items[:top_k]]
    print(f"  Выбрано топ-{len(top_items)} популярных товаров (покупки+просмотры)")

    # --- 5. Вывод топ-10 для анализа ---
    print("\nТоп-10 самых популярных товаров (покупки*3 + просмотры*1):")
    for i, (item_id, score) in enumerate(sorted_items[:10], 1):
         print(f"  {i}. Товар {item_id}: {score:.1f} баллов")

    # Очистка
    del item_purchases, item_views, combined_scores, sorted_items
    gc.collect()

    return top_items


In [13]:
# === Адаптированная функция 2: build_user_preferences (с tracker_df, батчами) ===
def build_user_preferences(orders_df, tracker_df, test_users, chunk_size=5_000_000):
    """
    Строит историю покупок и просмотров (предпочтения) для тестовых пользователей.
    Обрабатывает tracker_df по частям.
    """
    import pandas as pd
    from collections import defaultdict
    import gc

    print("Строим пользовательские предпочтения (покупки + просмотры, батчами)...")

    # --- 1. История покупок (все доставленные заказы) ---
    # Эта часть маленькая по сравнению с tracker, можно обработать целиком
    delivered_orders = orders_df[orders_df['last_status'] == 'delivered_orders']
    print(f"  Всего доставленных заказов: {len(delivered_orders):,}")
    user_purchased_items = delivered_orders.groupby('user_id')['item_id'].apply(list).to_dict()
    print(f"  Пользователей с покупками: {len(user_purchased_items):,}")
    del delivered_orders
    gc.collect()

    # --- 2. История просмотров (tracker_df) по частям ---
    print(f"  Собираем последние просмотры для тестовых пользователей из tracker_df (батчами)...")
    # Используем словарь, где ключ - user_id, значение - список (timestamp, item_id)
    # Это позволит потом отсортировать и взять последние 50
    temp_user_views = defaultdict(list)
    test_users_set = set(test_users) # Для быстрого поиска

    total_rows = len(tracker_df)
    num_chunks = (total_rows // chunk_size) + 1

    for i in range(num_chunks):
        start_row = i * chunk_size
        end_row = min((i + 1) * chunk_size, total_rows)
        chunk = tracker_df.iloc[start_row:end_row]

        # Фильтрация чанка: только просмотры и только тестовые пользователи
        filtered_chunk = chunk[
            (chunk['action_type'] == 'page_view') &
            (chunk['user_id'].isin(test_users_set))
        ][['user_id', 'item_id', 'timestamp']] # Берем только нужные колонки

        # Накапливаем просмотры
        for _, row in filtered_chunk.iterrows():
             # Сохраняем кортеж (timestamp, item_id) для последующей сортировки
             temp_user_views[row['user_id']].append((row['timestamp'], row['item_id']))

        del chunk, filtered_chunk
        if i % 10 == 0 or i == num_chunks - 1:
             print(f"    Обработано {i+1}/{num_chunks} частей tracker_df для просмотров")
        gc.collect()

    # --- 3. Обработка накопленных просмотров: сортировка и ограничение до 50 ---
    print("  Обрабатываем накопленные просмотры (сортировка, последние 50)...")
    user_viewed_items = {}
    for user_id, view_list in temp_user_views.items():
        # Сортировка по timestamp по убыванию (новые первые)
        view_list.sort(key=lambda x: x[0], reverse=True)
        # Берем первые 50 item_id
        user_viewed_items[user_id] = [item_id for _, item_id in view_list[:50]]

    del temp_user_views
    gc.collect()
    print(f"  Пользователей с просмотрами: {len(user_viewed_items):,}")

    # --- 4. Объединение предпочтений ---
    print("  Объединяем покупки и просмотры...")
    user_preferences = {}
    users_with_history = 0

    for user_id in test_users:
        prefs = set()
        # Добавляем купленные
        prefs.update(user_purchased_items.get(user_id, []))
        # Добавляем просмотренные
        prefs.update(user_viewed_items.get(user_id, []))

        user_preferences[user_id] = list(prefs)
        if prefs:
            users_with_history += 1

    print(f"  Построены предпочтения для {len(user_preferences):,} тестовых пользователей")
    print(f"  Пользователей с какой-либо историей (покупки/просмотры): {users_with_history:,}")

    # Очистка
    del user_purchased_items, user_viewed_items
    gc.collect()

    return user_preferences


In [15]:
# === Функция 3: generate_recommendations (без изменений) ===
def generate_recommendations(test_users, popular_items, user_preferences):
    """
    Генерирует топ-100 рекомендаций для каждого тестового пользователя,
    исключая товары из его расширенных предпочтений (покупки + просмотры).
    """
    print("Генерируем рекомендации...")
    import tqdm
    recommendations = {}

    for user_id in tqdm.tqdm(test_users, desc="Создание рекомендаций"):
        bought_set = set(user_preferences.get(user_id, []))
        available_items = [item for item in popular_items if item not in bought_set]

        user_recs = []
        if len(available_items) >= 100:
            user_recs = available_items[:100]
        else:
            user_recs = available_items
            items_needed = 100 - len(user_recs)
            additional_items = [item for item in popular_items if item not in user_recs][:items_needed]
            user_recs.extend(additional_items)

        recommendations[user_id] = user_recs[:100]

    print(f"  Сгенерированы рекомендации для {len(recommendations):,} пользователей")
    return recommendations


In [ ]:
# --- Запуск адаптированного пайплайна с обработкой батчами ---
print("\n--- Запуск адаптированного пайплайна (с tracker_df, батчами) ---")

# 1. Построение популярности
# Передаем tracker_df, функция сама разобьет его на части
popular_items = build_popularity_model(
    orders_df,
    tracker_df,
    period_start='2025-07-02',
    period_end='2025-07-15',
    top_k=100,
    chunk_size=5_000_000 # Настрой под свой объем ОЗУ
)

# 2. Построение пользовательских предпочтений
user_preferences = build_user_preferences(
    orders_df,
    tracker_df,
    test_users,
    chunk_size=5_000_000 # Настрой под свой объем ОЗУ
)

# 3. Генерация рекомендаций
recommendations = generate_recommendations(test_users, popular_items, user_preferences)



--- Запуск адаптированного пайплайна (с tracker_df, батчами) ---
Строим модель популярности (с учетом просмотров, батчами)...
  Отфильтровано заказов за период 2025-07-02 - 2025-07-15: 484,645
  Уникальных купленных товаров в периоде: 279,446
  Обрабатываем tracker_df по частям размером ~5,000,000 записей...
    Обработано 1/24 частей tracker_df
    Обработано 11/24 частей tracker_df
    Обработано 21/24 частей tracker_df
    Обработано 24/24 частей tracker_df
  Всего уникальных просмотренных товаров в периоде: 2,951,075
  Рассчитываем комбинированный рейтинг...
  Уникальных товаров с комбинированным рейтингом: 2,951,971
  Выбрано топ-100 популярных товаров (покупки+просмотры)

Топ-10 самых популярных товаров (покупки*3 + просмотры*1):
  1. Товар 12904245: 15677.0 баллов
  2. Товар 51974017: 12288.0 баллов
  3. Товар 207631139: 11808.0 баллов
  4. Товар 172018601: 11406.0 баллов
  5. Товар 26556597: 11070.0 баллов
  6. Товар 113070693: 10743.0 баллов
  7. Товар 165938954: 10628.0 балл